### Imports

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

### Loading Data

In [2]:
data_path = "datasets"
contract_path = os.path.join(data_path, "contract.csv")
internet_path = os.path.join(data_path, "internet.csv")
personal_path = os.path.join(data_path, "personal.csv")
phone_path = os.path.join(data_path, "phone.csv")

df_contract = pd.read_csv(contract_path)
df_internet = pd.read_csv(internet_path)
df_personal = pd.read_csv(personal_path)
df_phone = pd.read_csv(phone_path)

### Exploratory Data Analysis and Data Wrangling

#### contract dataframe

In [3]:
df_contract.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB


In [4]:
df_contract.sample(10)

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
6224,2330-PQGDQ,2015-11-01,No,One year,No,Bank transfer (automatic),84.20,4299.75
6118,9924-JPRMC,2014-02-01,No,Two year,Yes,Electronic check,118.20,8547.15
6157,6776-TLWOI,2019-07-01,2019-10-01 00:00:00,Month-to-month,No,Mailed check,19.85,64.55
3013,2065-MMKGR,2017-09-01,No,One year,Yes,Credit card (automatic),71.00,2080.1
5103,6421-SZVEM,2017-10-01,No,One year,Yes,Bank transfer (automatic),82.85,2320.8
5027,3247-ZVOUO,2019-01-01,2019-11-01 00:00:00,Month-to-month,No,Electronic check,85.55,851.75
4138,5597-GLBUC,2019-10-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,85.45,85.45
1487,5035-PGZXH,2015-06-01,No,One year,Yes,Electronic check,106.80,5914.4
4306,2931-SVLTV,2016-10-01,No,Month-to-month,Yes,Credit card (automatic),110.10,4469.1
4330,4013-TLDHQ,2018-05-01,2019-12-01 00:00:00,Month-to-month,Yes,Electronic check,78.25,1490.95


Things to check in general in this dataframe.
- Check for duplicate rows in the dataframe

Things to check within this dataframe;

- custormerID: if there are nulls or repeated IDs
- BeginDate: if there are nulls or incorrect dates (greater than today or earlier than a reasonable past date). Change format to pd.Datetime
- EndDate: if there are nulls or incorrect dates (earlier than BeginDate or later than today); transform "No" into pd.NaT value
- Type: check uniques in order to transform into a categorical variable
- PaperlessBilling: check uniques in order to transform into a categorical variable
- PaymentMethod: check uniques in order to transform into a categorical variable
- MonthlyCharges: check for nulls and incorrect values (negative or excessively high)
- TotalCharges: check for nulls and incorrect values (negative or excessively high)


##### General

In [5]:
# Check for nulls
df_contract.isnull().sum()

customerID          0
BeginDate           0
EndDate             0
Type                0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

In [6]:
# Check for duplicate rows in the contract dataframe
df_contract.duplicated().sum()

np.int64(0)

##### customerID

In [7]:
# Check for duplicate customerID in the contract dataframe
sum(df_contract["customerID"].value_counts() > 1)

0

In [8]:
# Check null or empty values in the contract dataframe


##### BeginDate

In [9]:
# Convert BeginDate to datetime format
df_contract["BeginDate"] = pd.to_datetime(df_contract["BeginDate"], errors="coerce", format="%Y-%m-%d")
df_contract.sample(10)

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
6636,3468-DRVQJ,2019-04-01,No,One year,No,Electronic check,70.30,676.15
5292,0756-MPZRL,2016-04-01,No,One year,No,Credit card (automatic),33.70,1537.85
3835,8194-PEEBY,2018-02-01,No,One year,No,Mailed check,19.90,533.5
2665,8166-ORCHU,2017-05-01,No,One year,Yes,Electronic check,93.55,3055.5
3229,5519-NPHVG,2018-11-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,94.20,1046.1
3090,2386-LAHRK,2019-09-01,2019-10-01 00:00:00,Month-to-month,Yes,Mailed check,53.50,53.5
4986,2694-CIUMO,2019-02-01,No,Month-to-month,Yes,Credit card (automatic),79.55,958.25
3785,8337-UPOAQ,2019-03-01,2020-01-01 00:00:00,Month-to-month,Yes,Electronic check,89.80,914.3
1552,2245-ADZFJ,2017-07-01,No,Two year,No,Bank transfer (automatic),80.55,2471.6
3649,0174-QRVVY,2014-03-01,No,Two year,No,Credit card (automatic),25.35,1847.55


##### EndDate

In [10]:
# Convert EndDate to datetime format
df_contract["EndDate"] = pd.to_datetime(df_contract["EndDate"], errors="coerce", format="%Y-%m-%d %H:%M:%S")
df_contract.sample(10)

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
6341,9330-IJWIO,2018-10-01,2019-11-01,Month-to-month,Yes,Electronic check,100.35,1358.85
5625,8963-MQVYN,2018-04-01,NaT,Month-to-month,Yes,Electronic check,20.55,469.85
1513,0661-XEYAN,2019-09-01,2019-10-01,Month-to-month,Yes,Mailed check,25.80,25.8
2953,2249-YPRNG,2018-04-01,2019-12-01,Month-to-month,Yes,Bank transfer (automatic),105.85,2239.65
2083,4897-QSUYC,2019-11-01,2019-12-01,Month-to-month,Yes,Mailed check,20.15,20.15
926,7074-IEVOJ,2019-09-01,2019-12-01,Month-to-month,Yes,Electronic check,79.40,205.05
2509,1926-QUZNN,2014-02-01,NaT,Two year,Yes,Bank transfer (automatic),25.25,1841.2
6707,2452-MRMZF,2014-02-01,NaT,Two year,Yes,Credit card (automatic),25.70,1937.4
94,9848-JQJTX,2014-02-01,NaT,Two year,Yes,Bank transfer (automatic),100.90,7459.05
1183,0887-WBJVH,2015-09-01,NaT,One year,Yes,Electronic check,93.45,4872.2


##### Type

In [11]:
# Check unique types
df_contract["Type"].unique()

array(['Month-to-month', 'One year', 'Two year'], dtype=object)

##### PaperlessBilling

In [12]:
# Check unique types
df_contract["PaperlessBilling"].unique()

array(['Yes', 'No'], dtype=object)

##### PaymentMethod

In [13]:
# Check unique types
df_contract["PaymentMethod"].unique()

array(['Electronic check', 'Mailed check', 'Bank transfer (automatic)',
       'Credit card (automatic)'], dtype=object)

##### MonthlyCharges

In [14]:
# check for negative or extreamly high values
min = df_contract[df_contract["MonthlyCharges"] < 0]
high = df_contract[df_contract["MonthlyCharges"] > 1000]
print("Negative MonthlyCharges:\n", min)
print("Extremely High MonthlyCharges:\n", high)

Negative MonthlyCharges:
 Empty DataFrame
Columns: [customerID, BeginDate, EndDate, Type, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges]
Index: []
Extremely High MonthlyCharges:
 Empty DataFrame
Columns: [customerID, BeginDate, EndDate, Type, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges]
Index: []


##### TotalCharges

In [15]:
# This column is str; it will be changed to numerical
df_contract["TotalCharges"] = pd.to_numeric(df_contract["TotalCharges"], errors="coerce")

# check if there are nulls
df_contract[df_contract["TotalCharges"].isna()]
idx = df_contract[df_contract["TotalCharges"].isna()].index

As seen, the "TotalCharges" column had some non-numeric values which were converted to NaN.

This probably is due the end of the timelapse of the dataframe, where some entries have not accumulated any charges yet.

Therefore, it might be reasonable to fill these NaN values with the same monthly value, assuming no charges have been accumulated yet; but prio to do so, I'll explore which is the most recent date in the dataframe

In [16]:
df_contract["BeginDate"].max()

Timestamp('2020-02-01 00:00:00')

In [ ]:
# Since my guess was correct: Replacing NaN values in the "TotalCharges" column with the value MonthlyCharges
df_contract["TotalCharges"].fillna(df_contract["MonthlyCharges"], inplace=True)
df_contract.iloc[idx]

C:\Users\Atmosfera\AppData\Local\Temp\ipykernel_8748\2833428116.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_contract["TotalCharges"].fillna(df_contract["MonthlyCharges"], inplace=True)


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
488,4472-LVYGI,2020-02-01,NaT,Two year,Yes,Bank transfer (automatic),52.55,52.55
753,3115-CZMZD,2020-02-01,NaT,Two year,No,Mailed check,20.25,20.25
936,5709-LVOEQ,2020-02-01,NaT,Two year,No,Mailed check,80.85,80.85
1082,4367-NUYAO,2020-02-01,NaT,Two year,No,Mailed check,25.75,25.75
1340,1371-DWPAZ,2020-02-01,NaT,Two year,No,Credit card (automatic),56.05,56.05
3331,7644-OMVMY,2020-02-01,NaT,Two year,No,Mailed check,19.85,19.85
3826,3213-VVOLG,2020-02-01,NaT,Two year,No,Mailed check,25.35,25.35
4380,2520-SGTTA,2020-02-01,NaT,Two year,No,Mailed check,20.00,20.00
5218,2923-ARZLG,2020-02-01,NaT,One year,Yes,Mailed check,19.70,19.70
6670,4075-WKNIU,2020-02-01,NaT,Two year,No,Mailed check,73.35,73.35


In [ ]:
# check for negative or extreamly high values
min = df_contract[df_contract["TotalCharges"] < 0]
high = df_contract[df_contract["TotalCharges"] > 1000]
print("Negative TotalCharges:\n", min)
print("Extremely High TotalCharges:\n", high)

### internet dataframe

In [ ]:
df_internet.info()

In [ ]:
df_internet.head()

### personal dataframe

In [ ]:
df_personal.info()

### phone dataframe

In [ ]:
df_phone.info()

#### Dataframes merge and target creation

This shall be made using customerID as common (anchor) for all dataframes

#### Conclusions of EDA

### Models Training

### Split data into training and testing sets

#### Functions for training and evaluation

#### Logistic Regression

#### Decision Tree

#### Random Forest

#### CatBoost

### Best Model Evaluation

In [ ]:
### Predictions

###